# Script 1: Run through the general pre-processing steps of Liver Project 
## By Alisha Aristel

## To get an understanding of what the Data actually is:

- one sample = spatial grid of pixels 
  - 1. Coordinates :
    - for every pixel:
      - x coordinate 
      - y coordinate
      - `coords R ^(N x 2) matrix of real numbers`
      - N = number of pixels
      - each row = one pixels location in tissue space 
      - ex: pixel x    y
      -      1.   120  45
      -      N.   87.   132
  - 2. Gene expression Matrix (X) :
    - for the same N pixels, you have expression values for G genes
    - rows = pixels
    - columns = genes
    - `X R^(N x G)`
    - `Xi,g = expression of gene g at pixel i`
  - 3. pixtot (total intensity):
    - for each pixel, you have 
      - `$pixtot_{i}$ = $\sum_{g=1}^{G}$ = $X_{i,g}$
      - this is a vector where pixtot R^N


so for pixel i:
- coords[i] -> where it is
- X[i, :] -> what gene it expresses
- pixtot[i] -> how strong the signal is overall

### Clipping
we clipped b/c the gene expression distrubtion is skewed and dominated by a few extreme pixels
- so for a gene g,
  - most pixels have small values
  - a few have massive values

Per-gene clipping
for each gene g, compute:
- high quantile= compute the 98th percentile:
  - qg = quantile0.98(X.,g)
    - 98% of pixels have expressions <= qg
    - 2 % of pixels are extreme high values
  - clip (cap) values
    - define a new matrix X' 
      - X'i,g = min(xi,g, qg)
      - so:
        - if a pixel has normal expression -> unchanged
        - if a pixel is an extreme hotspot-> capped

https://docs.astropy.org/en/latest/stats/robust.html

### Masking= what gets removed and why

- masking is pixel removal 
- maskpixi {0,1} 
- where:
- 1= bad pixel
- 0 = keep pixel
- the rules based on pixtot:
  - low-intensity pixels:
    - `pixtoti < Q0.02(pixtot)`
  - high-intensity pixels:
    - `pixtoti > Q0.99(pixtot)`
  
  

### Normalization 
normalize = convert raw expression into expression relative to a baseline
step 1: variance stabilization (https://www.geeksforgeeks.org/data-science/variance-stabilizing-transformation/)
- xi,g = sqrt(Xig)
- sqaure root: compresses large values and stabilizes variance. this is for poisson count data where variance equals the mean and less aggressive than log transformation 






In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import Normalize, ListedColormap
from matplotlib import cm
from matplotlib.patches import Patch
from statsmodels.nonparametric.smoothers_lowess import lowess
import pyreadr
from functools import reduce

from joblib import dump

## Pipeline fully 

### Block 0: CONFIG

#### list describing each sample 
these are all parallel list: 

1. FILENAME[i], GENDER[i], AGE[i]
2. FILENAMES = [ ... 13 filenames ... ]
3. GENDER = [...]
4. MUTANT = [...]
5. AGE = [...]
6. HANDLE = [...]
7. PLOTCOLOR = [...]
8. PLOTLTY = [...]

#### thesholds/parameters

#### output dir

In [24]:

DATADIR    = "/home/stat/nzh/team/aristela/nzhanglab/data/Laux_liver_visium/Data"


# files and metadata for RData loading
FILENAMES          = [
    "v44t19-391-a1_pixel_table_1000_genes_cropped.RData",
    "v44t19-344-a1_pixel_table_1000_genes_cropped.RData",
    "v44l24-326-a1_pixel_table_1000_genes_cropped.RData",
    "v44t19-284-a1_pixel_table_1000_genes_cropped.RData",
    "v44t19-391-d1_pixel_table_1000_genes_cropped.RData",
    "v44l24-326-d1_pixel_table_1000_genes_cropped.RData",
    "v44t19-344-d1_pixel_table_1000_genes_cropped.RData",
    "v44t19-284-d1_pixel_table_1000_genes_cropped.RData",
    "v13m13-378-a1_pixel_table_1000_genes_cropped.RData",
    "v13m13-380-a1_pixel_table_1000_genes_cropped.RData",
    "v12d05-320-d1_pixel_table_1000_genes_cropped.RData",
    "v13m13-380-d1_pixel_table_1000_genes_cropped.RData",
    "v12d05-320-c1_pixel_table_1000_genes_cropped.RData"
]
GENDER            = ["female", "female", "male", "male", "female", "male", "male", "male", "male", "male", "female", "female", "female"]
MUTANT             = ["wildtype", "mutant", "wildtype", "mutant", "wildtype", "mutant", "wildtype", "mutant", "wildtype", "wildtype", "wildtype", "wildtype", "mutant"]
AGE             = ["4.5 months", "4 months", "2.5 years", "4.5 months", "2.5 years", "4.5 months", "2.5 years", "4.5 months", "4 months", "4 months","4 months","2.6 years", "4 months" ]
HANDLE             = ["fwy", "fmy", "mwo", "mmy", "fwo", "mmy2", "mwo2", "mmy3", "mwy", "mwy2", "fwy2", "fwo2", "fmy2"]
PLOTCOLOR          = ["chartreuse4", "maroon1", "lightsalmon4", "darkorchid2", "orangered2", "darkorchid2", "lightsalmon4", "darkorchid2", "steelblue", "purple4", "seagreen4", "cyan4", "orangered2"]
PLOTLTY          = [1, 3, 2, 3, 2, 3, 2, 3, 4, 4, 5, 6, 4]



# clipping thresholds

CLIP_PCT  = 0.98   # per-gene clipping threshold (quantile on gene columns)
MASK_LOW  = 0.02   # low pixtot threshold (2%)
MASK_HIGH = 0.99   # high pixtot threshold (99%)
GENES_ENDOTHELIAL  = ["Cd31","Cd144","Vwf"]

# output dirs
CHECK_DIR          = "plots/clip_checks_updated"
SPATIAL_DIR        = "plots/mask_spatial_updated"
os.makedirs(CHECK_DIR, exist_ok=True)
os.makedirs(SPATIAL_DIR, exist_ok=True)


### Block 1: helper functions

### Block 1.1 read_txt
this reads the the data in and returns a list[str] gene names:

input:
filepath: str

output: 
list[str] - one string per line

In [29]:
def read_txt(filepath):
    with open(filepath, "r") as f:
        return [line.strip() for line in f]
    

### Block 1.2 normalize_data

Inputs:
- df = (N × G)
  - rows = pixels
  - columns = genes
  - values = expression counts / intensities
- housekeeping = [g₁, g₂, …, gₖ]

Output:
df_norm = (N × G) just normalized


how to use it:
mat_df = pd.DataFrame(d["mat_clipped"], columns=d["genes"])
d["matnorm"] = normalize_hk_sqrt(mat_df, genes_housekeeping)


In [ ]:
def normalize_data(df, housekeeping):
    
    df_sqrt = np.sqrt(df.copy())
   
    hk_present = [g for g in housekeeping if g in df_sqrt.columns]
   
    ctr_vals = df_sqrt[hk_present].sum(axis=1)

    df_norm = df_sqrt.div(ctr_vals, axis=0)
    
    return df_norm

### Block 2: loading gene list

In [ ]:
tab = pd.read_csv(os.path.join(DATADIR, "mfuzz.csv")).dropna(subset=["x"])
tab.columns = ["genename", "clusterid"]
genes_zonation_clusters = tab
genes_senescence_all = read_txt(os.path.join(DATADIR, "senescence_genes_20250307.txt"))
genes_zonation      = read_txt(os.path.join(DATADIR, "zonation_genes.txt"))
genes_fibroblast    = read_txt(os.path.join(DATADIR, "fibroblast_genes.txt"))


genes_midlobular    = ["Hamp","Hamp2","Ccnd1","Cyp8b1"]
genes_pericentral   = ["Cyp2e1","Gsta3","Cyp27a1","Mup17","Nt5e","Axin2","Cyp1a2","Oat","Gstm3","Axin2","Lgr5"]
genes_periportal    = ["Alb","Cyp2f2","Asl","Gls2","Cdh1","Cps1","Pck1","Sdhd"]
genes_housekeeping  = ["Gapdh","Actb","Vcl","L3mbtl2","Rbck1","Wdr55","Hprt"]
genes_immune        = ["Cd45","Cd68","Cd64","Cd3","Cd4","Cd8"]
genes_endothelial   = GENES_ENDOTHELIAL

genes_all = list(set(
    genes_zonation + genes_senescence_all + genes_fibroblast +
    genes_housekeeping + genes_immune + genes_endothelial +
    genes_midlobular + genes_periportal + genes_pericentral +
    genes_zonation_clusters["genename"].tolist()
))


### Block 3: loading all of the data in 
Placing it into a list called `dat` where each element is one `sample (so one .RData file)`
```
ex. dat = [sample1, sample2, sample3]
    so with each sample:
    - pixel coordinates (coords)
    - per-pixel total intensity(pixtot)
    - pixel x gene matrix (mat)
    - gene names (genes)
    - metadata like gender/mutant/age/handle
```

for i,fname in enumerate(FILENAMES) so it would do 
- pass 1: i=0, fename=FILENAMES[0]
- pass 2: i=1, fename=FILENAMES[1]
- step 2: would get to read in the data to `result` which for each sample would return datfm_inbox and pixtot
- step 3: i extract the 2 objects ( `datfm_inbox and pixtot`)
- step 4: separate gene expression from the coordniates and place it into a temp data 
- step 5: keep only numeric columns (gene expression) might contain some non-gene columns (strings)
- step 6: convert to a raw numpy matrix (which is a N, G) 
  -  N = number of pixels 
  -  G = number of gene columns
-  step 7: append a sample dict into the dat


In [ ]:
dat = [] # list

for i,fname in enumerate(FILENAMES):
    print(f'loading the data file{i+1}/{len(FILENAMES)}: {fname}') 
    result = pyreadr.read_r(os.path.join(DATADIR, fname))
    datfm_inbox = result.get("datfm_inbox")
    pixtot = result.get("pixtot")
    temp = datfm_inbox.drop(columns=["x", "y"], errors="ignore") 
    num_cols = temp.select_dtypes(include=[np.number]).columns.tolist() 
    mat = temp[num_cols].to_numpy()
    
    dat.append({
        "coords": datfm_inbox[["x", "y"]],
        "pixtot": pixtot.squeeze(),
        "mat" : mat, 
        "genes": num_cols,
        "gender": GENDER[i],
        "mutant": MUTANT[i],
        "age": AGE[i],
        "handle": HANDLE[i],
        "plotcolor": PLOTCOLOR[i % len(PLOTCOLOR)],
        "plotlty": PLOTLTY[i len(PLOTLTY)],
        "maskpix": None
    })
    
    

### Block 4: summary stats
- checks on the number of genes

In [ ]:

for sample in dat:
    print(f'{sample['handle']}: {len(sample['genes'])} genes')
    all_gene_sets = [set(sample['genes']) for sample in dat]
    common = set.intersection(*all_gene_sets)
    union  = set.union(*all_gene_sets)
    print(f"\nGenes in every sample: {len(common)}")
    print(f"Total unique genes:    {len(union)}")
    

### Block 5: Clipping and masking and plots

- step 1: pull things out of the sample dict (like handle, genes, pixtot, and matrix)
- step 2: turn it into a df
  - inputs 
    - mat (N x G)
    - genes list of length G
  - outputs
  - df (Nx G)
- step 3: clipping extreme values per gene 
  - gets the 98th percentile across pixels within this sample 
  - clips each gene coloumn using its own cutoff
- step 4: masking pixels using pixtot
  - compute low/high thresholds on pixtot and just mark it
- step 5: Histrogram QC plot (pixel-level )
- step 6: spitial maps
  - step 1 get the coordniates per sample
  - step 2:helper function: pivot scattered values into a grid
  - step 3: bluid four grids ( 0/1 space)
  - step 4: plot the 4 panel figure and save

In [ ]:
for sample in dat:
    handle = sample["handle"]
    genes = sample["genes"] # gene metadata
    pixtot = np.asarray(sample["pixtot"], dtype=float) ## pixel level
    mat = sample["mat"] # pixel x gene
    df = pd.DataFrame(mat, columns=genes)
    #### clipping
    clip_vals = df.quantile(CLIP_PCT)
    df_clipped = df.clip(upper=clip_vals, axis=1)
    sample["mat_clipped"] = df_clipped.values
    ### masking pixels 
    q_low = np.quantile(pixtot, MASK_LOW)
    q_high = np.quantile(pixtot, MASK_HIGH)
    pixel_low = pixtot < q_low
    pixel_high = pixtot > q_high
    maskpix = pixel_low | pixel_high 
    sample["maskpix"] = maskpix
    kept = ~maskpix
    high_only = pixel_high 
    low_only = pixel_low 
    # A) Histogram of pixtot by category
    fig, ax = plt.subplots(figsize=(6,4))
    ax.hist(pixtot[kept],      bins=50, alpha=0.6, label=f"kept ({kept.sum():,})")
    ax.hist(pixtot[high_only], bins=50, alpha=0.6, label=f"high ({high_only.sum():,})")
    ax.hist(pixtot[low_only],  bins=50, alpha=0.6, label=f"low ({low_only.sum():,})")
    ax.set(xlabel="total pixel intensity (pixtot)", ylabel="count",
           title=f"{handle} — pixtot by category")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(f"{CHECK_DIR}/{handle}_pixtot_maskcheck.png", dpi=300)
    plt.close(fig)
    ### B spital maps qc 
    coords=sample["coords"]
    ##
    def pivot(vals, fill=0):
           tmp = pd.DataFrame({"x": coords.x, "y": coords.y, "v": vals})
           return tmp.pivot_table(index="y", columns="x", values="v", fill_value=fill)
    piv_tot  = pivot(pixtot, fill=0)
    piv_high = pivot(high_only.astype(int), fill=np.nan)
    piv_low  = pivot(low_only.astype(int),  fill=np.nan)
    piv_keep = pivot(kept.astype(int),      fill=np.nan)
    fig, axes = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)
    
    ## plot total intensity (continous intensity map)
    im0 = axes[0].imshow(piv_tot.values, origin="lower", cmap="magma", interpolation="none")
    axes[0].set(title="Total intensity"); axes[0].axis("off")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="pixtot")
    # high mask
    axes[1].imshow(np.ma.masked_invalid(piv_high.values), origin="lower",
                   cmap=ListedColormap(["lightcyan","black"]), vmin=0, vmax=1, interpolation="none")
    axes[1].set(title=f"High mask (>{int(MASK_HIGH*100)}%)"); axes[1].axis("off")
    # low mask
    axes[2].imshow(np.ma.masked_invalid(piv_low.values), origin="lower",
                   cmap=ListedColormap(["lightcyan","black"]), vmin=0, vmax=1, interpolation="none")
    axes[2].set(title=f"Low mask (<{int(MASK_LOW*100)}%)"); axes[2].axis("off")
    ## kept
    axes[3].imshow(np.ma.masked_invalid(piv_keep.values), origin="lower",
                   cmap=ListedColormap(["lightgray","black"]), vmin=0, vmax=1, interpolation="none")
    axes[3].set(title="Kept pixels"); axes[3].axis("off")
    axes[3].legend(handles=[Patch(facecolor="black", label="True")],
                   loc="upper right", framealpha=0.7)
    plt.savefig(f"{SPATIAL_DIR}/{handle}_spatial_pixtot_masks.png", dpi=300)
    plt.close(fig)


    

### Block 6: Normalize data





In [ ]:
for i, sample in enumerate(dat):
    print(f"Normalizing sample {i+1} ({sample['handle']})")
    mat_df = pd.DataFrame(sample["mat_clipped"], columns=sample["genes"])
    matnorm = normalize_data(mat_df, genes_housekeeping)
    sample["matnorm"] = matnorm

### block 7: Save 

In [ ]:
out_path = os.path.join(DATADIR, "dat_after_norm.joblib")
dump(dat, out_path, compress=False)
print(f"All done!!! — data saved to {out_path}")
